# Animating Cycle Representatives

This tutorial shows the common animation workflows: a 2D filtration animation, the same animation with a barcode panel, a scalar measurement animated along a bar, and optional 3D exports.

The executable cells do not save files by default. Set `SAVE_OUTPUTS = True` in the export section to write tutorial media under `examples/example_figures/`.


In [ ]:
from pathlib import Path

import numpy as np
from persforest import PersistenceForest
from persforest.cycle_rep_vectorisations import signed_chain_edge_length


def sample_noisy_circle(n=120, noise=0.035, seed=4):
    rng = np.random.default_rng(seed)
    theta = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    theta = theta + rng.normal(scale=0.01, size=n)
    radius = 1.0 + rng.normal(scale=noise, size=n)
    return np.column_stack((radius * np.cos(theta), radius * np.sin(theta)))


def sample_noisy_sphere(n=90, noise=0.04, seed=8):
    rng = np.random.default_rng(seed)
    z = rng.uniform(-1.0, 1.0, n)
    theta = rng.uniform(0.0, 2.0 * np.pi, n)
    radius = 1.0 + rng.normal(scale=noise, size=n)
    xy = np.sqrt(1.0 - z * z)
    sphere = np.column_stack((xy * np.cos(theta), xy * np.sin(theta), z))
    return radius[:, None] * sphere


## Build the example forest

The small noisy circle has one long bar and keeps animation generation fast.


In [ ]:
points = sample_noisy_circle(n=100)
forest = PersistenceForest(points)

forest.plot_at_filtration(
    0.45,
    min_bar_length=0.05,
    coloring="bars",
    vertex_size=7,
)


## 2D filtration animation

With `filename=None`, `animate_filtration` returns a Matplotlib animation object and figure without saving to disk.


In [ ]:
anim, fig = forest.animate_filtration(
    filename=None,
    frames=30,
    t_min=0.0,
    t_max=0.8,
    fps=15,
    dpi=90,
    figsize=(4, 4),
    coloring="bars",
    show_cycles=True,
    show_complex=True,
    filtration_kwargs={"vertex_size": 5, "min_bar_length": 0.05},
)

anim


## 2D animation with a barcode panel

Set `with_barcode=True` to show the barcode and a moving filtration marker next to the point cloud.


In [ ]:
anim_with_barcode, fig_with_barcode = forest.animate_filtration(
    filename=None,
    with_barcode=True,
    frames=30,
    t_min=0.0,
    t_max=0.8,
    fps=15,
    dpi=90,
    figsize=(8, 4),
    panel_width_ratios=(3.0, 1.6),
    panel_spacing=0.08,
    coloring="bars",
    filtration_kwargs={"vertex_size": 5, "min_bar_length": 0.05},
    barcode_kwargs={"min_bar_length": 0.01},
)

anim_with_barcode


## Animate a measurement along a bar

`animate_barcode_measurement` uses a cycle functional such as `signed_chain_edge_length` and shows how that scalar changes along a selected bar. If no bar is supplied, the longest bar is used.


In [ ]:
measurement_anim, measurement_fig = forest.animate_barcode_measurement(
    cycle_func=signed_chain_edge_length,
    signed=False,
    frames=30,
    t_min=0.0,
    t_max=1.4,
    fps=15,
    dpi=90,
    total_figsize=(8, 4),
    filtration_kwargs={"vertex_size": 5, "min_bar_length": 0.05},
    measurement_kwargs={"color": "C1", "linewidth": 2.0},
)

measurement_anim


## Optional 2D export

Run this cell with `SAVE_OUTPUTS = True` when you want MP4 files. File names use the `tutorial_` prefix.


In [ ]:
SAVE_OUTPUTS = False
FIG_DIR = Path("examples") / "example_figures"

if SAVE_OUTPUTS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    forest.animate_filtration(
        filename=str(FIG_DIR / "tutorial_2d_filtration_with_barcode.mp4"),
        with_barcode=True,
        frames=80,
        t_min=0.0,
        t_max=0.8,
        fps=20,
        dpi=140,
        figsize=(8, 4),
        coloring="bars",
        filtration_kwargs={"vertex_size": 5, "min_bar_length": 0.05},
    )


## 3D animation and export choices

For 3D, use Plotly for quick interactive inspection. MP4 and HTML exports are available through `animate_filtration`; the export calls are optional because they write files and may require ffmpeg for MP4.


In [ ]:
points_3d = sample_noisy_sphere(n=80)
forest_3d = PersistenceForest(points_3d)

preview_3d = forest_3d.plot_filtration_interactive(
    filt_max=1.4,
    min_bar_length=0.1,
    resolution=16,
    vertex_size=2,
    cycle_opacity=0.75,
    height=650,
    show=False,
)

preview_3d


In [ ]:
if SAVE_OUTPUTS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

    forest_3d.animate_filtration(
        filename=str(FIG_DIR / "tutorial_3d_filtration.mp4"),
        format="mp4",
        with_barcode=True,
        frames=80,
        fps=20,
        dpi=140,
        coloring="forest",
        camera_mode="orbit",
        filtration_kwargs={"min_bar_length": 0.1, "vertex_size": 2},
    )

    forest_3d.animate_filtration(
        filename=str(FIG_DIR / "tutorial_3d_filtration.html"),
        format="html",
        frames=60,
        coloring="forest",
        filtration_kwargs={"min_bar_length": 0.1, "vertex_size": 2},
    )
